In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))

%load_ext autoreload
%autoreload 2

In [5]:
from src.load import run_full_load

run_full_load()

2026-08-04 00:33:14,727 | INFO | All warehouse tables truncated and reset.
2026-08-04 00:33:14,856 | INFO | Loaded 861 rows into dim_driver
2026-08-04 00:33:14,903 | INFO | Loaded 212 rows into dim_constructor
2026-08-04 00:33:14,952 | INFO | Loaded 77 rows into dim_circuit
2026-08-04 00:33:14,988 | INFO | Loaded 139 rows into dim_status
2026-08-04 00:33:15,229 | INFO | Loaded 1125 rows into dim_race
2026-08-04 00:33:15,231 | INFO | All dimension tables loaded.
2026-08-04 00:33:20,628 | INFO | Loaded 26759 rows into fact_race_results
2026-08-04 00:33:22,372 | INFO | Loaded 10494 rows into fact_qualifying
2026-08-04 00:33:22,374 | INFO | All fact tables loaded.


In [3]:
import pandas as pd
from src.db import engine

for table in ["dim_driver", "dim_constructor", "dim_circuit", "dim_status", "dim_race"]:
    count = pd.read_sql(f"SELECT COUNT(*) FROM {table}", engine)
    print(f"{table}: {count.iloc[0,0]} rows")

dim_driver: 861 rows
dim_constructor: 212 rows
dim_circuit: 77 rows
dim_status: 139 rows
dim_race: 1125 rows


In [4]:
from src.load import run_full_load

run_full_load()

2026-08-04 00:30:51,187 | INFO | All warehouse tables truncated and reset.
2026-08-04 00:30:52,944 | INFO | Loaded 861 rows into dim_driver
2026-08-04 00:30:53,011 | INFO | Loaded 212 rows into dim_constructor
2026-08-04 00:30:53,071 | INFO | Loaded 77 rows into dim_circuit
2026-08-04 00:30:53,117 | INFO | Loaded 139 rows into dim_status
2026-08-04 00:30:53,269 | INFO | Loaded 1125 rows into dim_race
2026-08-04 00:30:53,270 | INFO | All dimension tables loaded.
2026-08-04 00:30:59,137 | INFO | Loaded 26759 rows into fact_race_results
2026-08-04 00:31:01,125 | INFO | Loaded 10494 rows into fact_qualifying
2026-08-04 00:31:01,127 | INFO | All fact tables loaded.


In [6]:
for table in ["dim_driver", "dim_constructor", "dim_circuit", "dim_status", "dim_race", "fact_race_results", "fact_qualifying"]:
    count = pd.read_sql(f"SELECT COUNT(*) FROM {table}", engine)
    print(f"{table}: {count.iloc[0,0]} rows")

dim_driver: 861 rows
dim_constructor: 212 rows
dim_circuit: 77 rows
dim_status: 139 rows
dim_race: 1125 rows
fact_race_results: 26759 rows
fact_qualifying: 10494 rows


In [7]:
orphan_check = pd.read_sql("""
    SELECT COUNT(*) 
    FROM fact_race_results f
    LEFT JOIN dim_driver d ON f.driver_key = d.driver_key
    WHERE d.driver_key IS NULL
""", engine)
print(f"Orphaned driver references in fact_race_results: {orphan_check.iloc[0,0]}")

Orphaned driver references in fact_race_results: 0


In [8]:
sample = pd.read_sql("""
    SELECT d.forename, d.surname, r.year, r.name AS race_name, f.position, f.points
    FROM fact_race_results f
    JOIN dim_driver d ON f.driver_key = d.driver_key
    JOIN dim_race r ON f.race_key = r.race_key
    WHERE d.surname = 'Hamilton'
    ORDER BY r.year, r.round
    LIMIT 10
""", engine)
sample

,forename,surname,year,race_name,position,points
0,Duncan,Hamilton,1951,British Grand Prix,12.0,0.0
1,Duncan,Hamilton,1951,German Grand Prix,NaN,0.0
2,Duncan,Hamilton,1952,British Grand Prix,NaN,0.0
3,Duncan,Hamilton,1952,Dutch Grand Prix,7.0,0.0
4,Duncan,Hamilton,1953,British Grand Prix,NaN,0.0
5,Lewis,Hamilton,2007,Australian Grand Prix,3.0,6.0
6,Lewis,Hamilton,2007,Malaysian Grand Prix,2.0,8.0
7,Lewis,Hamilton,2007,Bahrain Grand Prix,2.0,8.0
8,Lewis,Hamilton,2007,Spanish Grand Prix,2.0,8.0
9,Lewis,Hamilton,2007,Monaco Grand Prix,2.0,8.0
